# 02. ROSE and LROM Cross-Section Comparison

In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import Any
import platform
import sys
import time

import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from numba import njit

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "lrom_legacy").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scipy.special
if not hasattr(scipy.special, "sph_harm") and hasattr(scipy.special, "sph_harm_y"):
    scipy.special.sph_harm = (
        lambda m, n, theta, phi: scipy.special.sph_harm_y(n, m, phi, theta)
    )

import rose
import lrom_legacy.v2_0 as lrom

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})


## 1. Physical Problem and Optical-Potential Parameters

In [2]:
TARGET = (40, 20)
PROJECTILE = (1, 0)
LAB_ENERGY = 14.1
L_MAX = 3
MESH_SIZE = 600
N_TRAIN = 200
N_TEST = 100
HALF_WIDTH = 0.20
ANGLES_DEG = np.arange(1.0, 180.0, 1.0)
ANGLES_RAD = np.deg2rad(ANGLES_DEG)
BASIS_SIZES = (4, 6, 8)
ROSE_EIM_SIZES = (4, 8, 12)
LROM_PREDICTOR_COUNTS = (4, 8, 12)
DEFAULT_BASIS_SIZE = 6
DEFAULT_COMPRESSION_SIZE = 8
TIMING_REPEATS = 3
TIMING_INNER_LOOPS = 20
SEED = 1204
ERROR_DENOMINATOR_FLOOR = 1e-12
PLOTTING_FLOOR = 1e-6


## 2. Shared Training and Testing Samples

In [ ]:
def parameter_dicts(
    rows: np.ndarray,
    parameter_names: tuple[str, ...],
) -> list[dict[str, float]]:
    return [
        {name: float(value) for name, value in zip(parameter_names, row)}
        for row in rows
    ]


def pointwise_relative_error(
    predicted: np.ndarray,
    reference: np.ndarray,
    denominator_floor: float,
) -> np.ndarray:
    denominator = np.maximum(np.abs(reference), denominator_floor)
    return np.abs(predicted - reference) / denominator


def summarize_relative_error(
    predicted: np.ndarray,
    reference: np.ndarray,
    denominator_floor: float,
) -> dict[str, np.ndarray]:
    pointwise = pointwise_relative_error(
        predicted,
        reference,
        denominator_floor,
    )
    return {
        "median_over_angle_error": np.median(pointwise, axis=1),
        "maximum_over_angle_error": np.max(pointwise, axis=1),
    }


def time_lrom_predictions(
    model: Any,
    cases: list[dict[str, float]],
    repeats: int,
    inner_loops: int,
) -> np.ndarray:
    model.predict(parameters=cases[:1], reconstruct_wavefunctions=False)
    seconds = []
    for case in cases:
        measurements = []
        for _ in range(repeats):
            start = time.perf_counter_ns()
            for _ in range(inner_loops):
                model.predict(
                    parameters=case,
                    reconstruct_wavefunctions=False,
                )
            measurements.append(
                (time.perf_counter_ns() - start) / (1e9 * inner_loops)
            )
        seconds.append(min(measurements))
    return np.asarray(seconds)


def time_rose_predictions(
    model: Any,
    rows: np.ndarray,
    repeats: int,
    inner_loops: int,
    smatrix_function: Callable[[Any, np.ndarray], tuple[np.ndarray, np.ndarray]],
    cross_section_function: Callable[
        [Any, np.ndarray, np.ndarray, np.ndarray, np.ndarray],
        np.ndarray,
    ],
    angles_rad: np.ndarray,
) -> np.ndarray:
    warm_splus, warm_sminus = smatrix_function(model, rows[0])
    cross_section_function(
        model,
        rows[0],
        warm_splus,
        warm_sminus,
        angles_rad,
    )
    seconds = []
    for row in rows:
        measurements = []
        for _ in range(repeats):
            start = time.perf_counter_ns()
            for _ in range(inner_loops):
                splus, sminus = smatrix_function(model, row)
                cross_section_function(
                    model,
                    row,
                    splus,
                    sminus,
                    angles_rad,
                )
            measurements.append(
                (time.perf_counter_ns() - start) / (1e9 * inner_loops)
            )
        seconds.append(min(measurements))
    return np.asarray(seconds)


In [ ]:
def build_sampled_study(
    target: tuple[int, int],
    projectile: tuple[int, int],
    lab_energy: float,
    l_max: int,
    half_width: float,
    training_size: int,
    testing_size: int,
    mesh_size: int,
    seed: int,
) -> tuple[
    Any,
    dict[str, float],
    dict[str, tuple[float, float]],
    tuple[str, ...],
    np.ndarray,
    np.ndarray,
    tuple[str, ...],
    tuple[str, ...],
]:
    emulator = lrom.LROM(
        target=target,
        projectile=projectile,
        lab_energy=lab_energy,
        l=tuple(range(l_max + 1)),
        potential="full_woods-saxon",
    )
    central = dict(emulator.central_parameters)
    ranges = {
        name: tuple(
            sorted(
                (
                    (1.0 - half_width) * value,
                    (1.0 + half_width) * value,
                )
            )
        )
        for name, value in central.items()
    }
    emulator.sampling(
        training_ranges=ranges,
        testing_ranges=ranges,
        training_size=training_size,
        testing_size=testing_size,
        mesh_size=mesh_size,
        strategy="latin_hypercube",
        seed=seed,
        high_fidelity_solver="runge_kutta",
        solver_options={"rk_tols": (1e-9, 1e-9)},
    )
    design = emulator.samples.design
    return (
        emulator,
        central,
        ranges,
        emulator.parameter_names,
        design.training.values.copy(),
        design.testing.values.copy(),
        design.training.case_ids,
        design.testing.case_ids,
    )


(
    emulator,
    central,
    ranges,
    parameter_names,
    train_rows,
    test_rows,
    train_ids,
    test_ids,
) = build_sampled_study(
    TARGET,
    PROJECTILE,
    LAB_ENERGY,
    L_MAX,
    HALF_WIDTH,
    N_TRAIN,
    N_TEST,
    MESH_SIZE,
    SEED,
)

assert train_rows.shape == (N_TRAIN, len(parameter_names))
assert test_rows.shape == (N_TEST, len(parameter_names))
assert not any(np.array_equal(a, b) for a in train_rows for b in test_rows)
for column, name in enumerate(parameter_names):
    lower, upper = ranges[name]
    assert np.all((lower <= train_rows[:, column]) & (train_rows[:, column] <= upper))
    assert np.all((lower <= test_rows[:, column]) & (test_rows[:, column] <= upper))

LS_LABEL = "LS-projected cross section"


## 3. Potential Variation and LROM Predictor Locations

In [ ]:
def predictor_radius_figure(
    emulator: Any,
    central: dict[str, float],
    ranges: dict[str, tuple[float, float]],
    parameter_names: tuple[str, ...],
    training_rows: np.ndarray,
    testing_rows: np.ndarray,
    predictor_count: int,
    minimum_radius: float,
    l_max: int,
) -> tuple[dict[Any, Any], pd.DataFrame, Figure]:
    radius_mesh = emulator.samples.mesh.radius
    radius_mask = radius_mesh >= 0.2
    plot_radius = radius_mesh[radius_mask]
    center_vector = np.asarray([central[name] for name in parameter_names])
    predictors = lrom.build_effective_interaction_predictors(
        full_order_models=emulator.samples.full_order_models,
        rho=emulator.samples.mesh.rho,
        radius=radius_mesh,
        central_values=center_vector,
        training_values=training_rows,
        testing_values=testing_rows,
        predictor_count=predictor_count,
        minimum_radius=minimum_radius,
    )
    range_scale = np.asarray(
        [ranges[name][1] - ranges[name][0] for name in parameter_names]
    )
    distance = np.linalg.norm(
        (testing_rows - center_vector[np.newaxis, :])
        / range_scale[np.newaxis, :],
        axis=1,
    )
    ranked = np.argsort(distance)
    sample_indices = ranked[
        np.linspace(0, len(ranked) - 1, 12, dtype=int)
    ]
    potential_rows = np.vstack(
        [center_vector, testing_rows[sample_indices]]
    )

    fig, axes = plt.subplots(2, 2, figsize=(12.0, 7.0), sharex=True)
    for row in potential_rows:
        central_part = lrom.full_woods_saxon(radius_mesh, row)
        spin_orbit_part = lrom.full_woods_saxon_spin_orbit(
            radius_mesh,
            row,
        )
        axes[0, 0].plot(
            plot_radius,
            central_part.real[radius_mask],
            alpha=0.55,
        )
        axes[0, 1].plot(
            plot_radius,
            central_part.imag[radius_mask],
            alpha=0.55,
        )
        l_minus_part = central_part - (l_max + 1) * spin_orbit_part
        l_plus_part = central_part + l_max * spin_orbit_part
        axes[1, 0].plot(
            plot_radius,
            l_minus_part.real[radius_mask],
            alpha=0.55,
        )
        axes[1, 1].plot(
            plot_radius,
            l_plus_part.real[radius_mask],
            alpha=0.55,
        )

    channel_colors = plt.cm.viridis(
        np.linspace(0.08, 0.92, len(predictors))
    )
    predictor_rows = []
    for color, (channel, state) in zip(
        channel_colors,
        predictors.items(),
    ):
        for radius in state.selected_radii:
            for ax in axes.flat:
                ax.axvline(radius, color=color, alpha=0.45, lw=0.9)
        predictor_rows.append(
            {
                "channel": str(channel),
                "selected radii [fm]": np.round(
                    state.selected_radii,
                    4,
                ),
            }
        )

    axes[0, 0].set_title("l=0 central: real")
    axes[0, 1].set_title("l=0 central: imaginary")
    axes[1, 0].set_title("l=3, j=l-1/2: real")
    axes[1, 1].set_title("l=3, j=l+1/2: real")
    for ax in axes[1]:
        ax.set_xlabel("r [fm]")
    for ax in axes[:, 0]:
        ax.set_ylabel("potential [MeV]")
    axes[0, 1].legend(
        handles=[
            Line2D([], [], color=color, label=f"channel {channel}")
            for color, channel in zip(channel_colors, predictors)
        ],
        fontsize=7,
        ncol=2,
    )
    fig.suptitle("Channel effective-interaction predictor radii")
    fig.tight_layout()
    return predictors, pd.DataFrame(predictor_rows), fig


# FIGURE: potential-predictor-rainbows
section3_predictor, predictor_radius_table, figure = predictor_radius_figure(
    emulator,
    central,
    ranges,
    parameter_names,
    train_rows,
    test_rows,
    DEFAULT_COMPRESSION_SIZE,
    0.5,
    L_MAX,
)
assert all(
    np.all(state.selected_radii >= 0.5)
    for state in section3_predictor.values()
)
plt.show()
display(predictor_radius_table)


## 4. Equal-Basis ROSE and LROM Emulators

In [ ]:
@njit
def bench_ws(r, radius, diffuseness):
    return 1.0 / (1.0 + np.exp((r - radius) / diffuseness))


@njit
def bench_ws_prime(r, radius, diffuseness):
    ex = np.exp((r - radius) / diffuseness)
    return -(ex / diffuseness) / (1.0 + ex) ** 2


@njit
def bench_full_ws(r, alpha):
    vv, wv, wd, _vso, rv, rd, _rso, av, ad, _aso = alpha
    return (
        -vv * bench_ws(r, rv, av)
        - 1j * wv * bench_ws(r, rv, av)
        + 4j * ad * wd * bench_ws_prime(r, rd, ad)
    )


@njit
def bench_full_ws_so(r, alpha, ldots):
    _vv, _wv, _wd, vso, _rv, _rd, rso, _av, _ad, aso = alpha
    return vso / 139.57039**2 * ldots * bench_ws_prime(r, rso, aso) / r


def build_rose_bases(
    interaction: Any,
    n_phi: int,
    emulator: Any,
    rho_mesh: np.ndarray,
) -> list[list[Any]]:
    bases = []
    for ell, interaction_row in enumerate(interaction.interactions):
        row_bases = []
        for spin_index, _ in enumerate(interaction_row):
            key = ell if len(interaction_row) == 1 else (ell, spin_index)
            model = emulator.samples.full_order_models[key]
            free_reference = np.asarray(
                [
                    rose.free_solutions.phi_free(
                        float(s),
                        ell,
                        emulator.kinematics.eta,
                    )
                    for s in rho_mesh
                ],
                dtype=np.complex128,
            )
            row_bases.append(
                rose.basis.CustomBasis(
                    solutions=np.asarray(
                        emulator.samples.training_wavefunctions[key],
                        dtype=np.complex128,
                    ).T.copy(),
                    phi_0=free_reference,
                    rho_mesh=rho_mesh,
                    n_basis=n_phi,
                    solver=model.solver,
                    subtract_phi0=True,
                    use_svd=True,
                    center=False,
                    scale=False,
                )
            )
        bases.append(row_bases)
    return bases


def build_rose_emulator(
    interaction: Any,
    n_phi: int,
    emulator: Any,
    rho_mesh: np.ndarray,
    l_max: int,
    angles_rad: np.ndarray,
    s0: float,
) -> Any:
    return rose.ScatteringAmplitudeEmulator(
        interaction,
        build_rose_bases(interaction, n_phi, emulator, rho_mesh),
        l_max=l_max,
        angles=angles_rad,
        s_0=s0,
        Smatrix_abs_tol=1e-8,
        initialize_emulator=True,
    )


def exact_smatrix_all_channels(
    sae: Any,
    row: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    splus = np.empty(len(sae.rbes), dtype=np.complex128)
    sminus = np.empty_like(splus)
    for ell, rbe_row in enumerate(sae.rbes):
        splus[ell] = rbe_row[0].basis.solver.smatrix(row)
        sminus[ell] = (
            splus[ell]
            if ell == 0
            else rbe_row[1].basis.solver.smatrix(row)
        )
    return splus, sminus


def emulated_smatrix_all_channels(
    sae: Any,
    row: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    splus = np.empty(len(sae.rbes), dtype=np.complex128)
    sminus = np.empty_like(splus)
    for ell, rbe_row in enumerate(sae.rbes):
        splus[ell] = rbe_row[0].S_matrix_element(row)
        sminus[ell] = (
            splus[ell]
            if ell == 0
            else rbe_row[1].S_matrix_element(row)
        )
    return splus, sminus


def cross_section_from_smatrix(
    sae: Any,
    parameters: np.ndarray,
    splus: np.ndarray,
    sminus: np.ndarray,
    angles_rad: np.ndarray,
) -> np.ndarray:
    if not np.array_equal(sae.angles, angles_rad):
        raise ValueError("ROSE angle cache does not match the study grid.")
    return sae.calculate_xs(splus, sminus, parameters).dsdo


def build_rose_emulators(
    basis_sizes: tuple[int, ...],
    eim_sizes: tuple[int, ...],
    l_max: int,
    parameter_count: int,
    emulator: Any,
    training_rows: np.ndarray,
    rho_mesh: np.ndarray,
    angles_rad: np.ndarray,
    s0: float,
    coordinate_potential: Callable[..., complex],
    spin_orbit_potential: Callable[..., complex],
) -> dict[tuple[int, int], Any]:
    emulators = {}
    for n_phi in basis_sizes:
        for n_u in eim_sizes:
            interaction = rose.InteractionEIMSpace(
                l_max=l_max,
                coordinate_space_potential=coordinate_potential,
                spin_orbit_term=spin_orbit_potential,
                n_theta=parameter_count,
                mu=emulator.kinematics.mu,
                energy=emulator.kinematics.e_com,
                is_complex=True,
                training_info=training_rows,
                explicit_training=True,
                n_basis=n_u,
                rho_mesh=rho_mesh,
            )
            emulators[(n_phi, n_u)] = build_rose_emulator(
                interaction,
                n_phi,
                emulator,
                rho_mesh,
                l_max,
                angles_rad,
                s0,
            )
    return emulators


def evaluate_fom_cross_sections(
    reference_sae: Any,
    training_rows: np.ndarray,
    testing_rows: np.ndarray,
    angles_rad: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    def evaluate(rows: np.ndarray) -> np.ndarray:
        return np.asarray(
            [
                cross_section_from_smatrix(
                    reference_sae,
                    row,
                    *exact_smatrix_all_channels(reference_sae, row),
                    angles_rad,
                )
                for row in rows
            ]
        )

    return evaluate(training_rows), evaluate(testing_rows)


def comparison_result(
    train_xs: np.ndarray,
    test_xs: np.ndarray,
    fom_train_xs: np.ndarray,
    fom_test_xs: np.ndarray,
    denominator_floor: float,
    test_seconds: np.ndarray,
) -> dict[str, np.ndarray]:
    train_error = summarize_relative_error(
        train_xs,
        fom_train_xs,
        denominator_floor,
    )
    test_error = summarize_relative_error(
        test_xs,
        fom_test_xs,
        denominator_floor,
    )
    return {
        "train_xs": train_xs,
        "test_xs": test_xs,
        "train_median_over_angle_error": train_error["median_over_angle_error"],
        "test_median_over_angle_error": test_error["median_over_angle_error"],
        "train_maximum_over_angle_error": train_error[
            "maximum_over_angle_error"
        ],
        "test_maximum_over_angle_error": test_error[
            "maximum_over_angle_error"
        ],
        "test_seconds": test_seconds,
    }


def evaluate_old_lrom(
    emulator: Any,
    basis_size: int,
    predictor_count: int,
    angles_deg: np.ndarray,
    test_cases: list[dict[str, float]],
    fom_test_xs: np.ndarray,
    denominator_floor: float,
    timing_repeats: int,
    timing_inner_loops: int,
) -> dict[tuple[int, int], dict[str, np.ndarray]]:
    emulator.train(
        basis_size=basis_size,
        predictor="potential",
        predictor_count=predictor_count,
        observable="cross_section",
        angles_degrees=angles_deg,
    )
    seconds = time_lrom_predictions(
        emulator,
        test_cases,
        timing_repeats,
        timing_inner_loops,
    )
    emulator.predict(
        parameters=test_cases,
        reconstruct_wavefunctions=False,
    )
    test_xs = emulator.predictions.cross_sections.values.copy()
    error = summarize_relative_error(
        test_xs,
        fom_test_xs,
        denominator_floor,
    )
    return {
        (basis_size, predictor_count): {
            "test_xs": test_xs,
            "test_median_over_angle_error": error[
                "median_over_angle_error"
            ],
            "test_maximum_over_angle_error": error[
                "maximum_over_angle_error"
            ],
            "test_seconds": seconds,
        }
    }


def evaluate_ls_oracle(
    emulator: Any,
    training_rows: np.ndarray,
    testing_rows: np.ndarray,
    fom_train_xs: np.ndarray,
    fom_test_xs: np.ndarray,
    denominator_floor: float,
) -> dict[str, np.ndarray]:
    train_coordinates = {
        channel: lrom.project_coordinates(
            basis=emulator.basis[channel],
            wavefunctions=emulator.samples.training_wavefunctions[channel],
        )
        for channel in emulator.basis
    }
    test_coordinates = {
        channel: lrom.project_coordinates(
            basis=emulator.basis[channel],
            wavefunctions=emulator.samples.testing_wavefunctions[channel],
        )
        for channel in emulator.basis
    }
    _, train_state = lrom._cross_section_prediction(
        emulator=emulator,
        values=training_rows,
        coefficients=train_coordinates,
    )
    _, test_state = lrom._cross_section_prediction(
        emulator=emulator,
        values=testing_rows,
        coefficients=test_coordinates,
    )
    train_xs = train_state.values.copy()
    test_xs = test_state.values.copy()
    train_error = summarize_relative_error(
        train_xs,
        fom_train_xs,
        denominator_floor,
    )
    test_error = summarize_relative_error(
        test_xs,
        fom_test_xs,
        denominator_floor,
    )
    return {
        "train_xs": train_xs,
        "test_xs": test_xs,
        "train_median_over_angle_error": train_error[
            "median_over_angle_error"
        ],
        "test_median_over_angle_error": test_error[
            "median_over_angle_error"
        ],
        "train_maximum_over_angle_error": train_error[
            "maximum_over_angle_error"
        ],
        "test_maximum_over_angle_error": test_error[
            "maximum_over_angle_error"
        ],
    }


def evaluate_lrom_grid(
    emulator: Any,
    basis_sizes: tuple[int, ...],
    predictor_counts: tuple[int, ...],
    default_config: tuple[int, int],
    angles_deg: np.ndarray,
    training_rows: np.ndarray,
    testing_rows: np.ndarray,
    train_cases: list[dict[str, float]],
    test_cases: list[dict[str, float]],
    fom_train_xs: np.ndarray,
    fom_test_xs: np.ndarray,
    denominator_floor: float,
    timing_repeats: int,
    timing_inner_loops: int,
) -> tuple[
    dict[tuple[int, int], dict[str, np.ndarray]],
    dict[int, dict[str, np.ndarray]],
    Any,
]:
    results = {}
    ls_results = {}
    default_predictor = None
    for n_phi in basis_sizes:
        for predictor_count in predictor_counts:
            emulator.train(
                basis_size=n_phi,
                predictor="effective-interaction",
                predictor_count=predictor_count,
                observable="cross_section",
                angles_degrees=angles_deg,
            )
            seconds = time_lrom_predictions(
                emulator,
                test_cases,
                timing_repeats,
                timing_inner_loops,
            )
            emulator.predict(
                parameters=train_cases,
                reconstruct_wavefunctions=False,
            )
            train_xs = emulator.predictions.cross_sections.values.copy()
            emulator.predict(
                parameters=test_cases,
                reconstruct_wavefunctions=False,
            )
            test_xs = emulator.predictions.cross_sections.values.copy()
            results[(n_phi, predictor_count)] = comparison_result(
                train_xs,
                test_xs,
                fom_train_xs,
                fom_test_xs,
                denominator_floor,
                seconds,
            )
            if (n_phi, predictor_count) == default_config:
                default_predictor = emulator.predictors
            if predictor_count == predictor_counts[0]:
                ls_results[n_phi] = evaluate_ls_oracle(
                    emulator,
                    training_rows,
                    testing_rows,
                    fom_train_xs,
                    fom_test_xs,
                    denominator_floor,
                )
    return results, ls_results, default_predictor


def evaluate_rose_grid(
    emulators: dict[tuple[int, int], Any],
    training_rows: np.ndarray,
    testing_rows: np.ndarray,
    angles_rad: np.ndarray,
    fom_train_xs: np.ndarray,
    fom_test_xs: np.ndarray,
    denominator_floor: float,
    timing_repeats: int,
    timing_inner_loops: int,
) -> dict[tuple[int, int], dict[str, np.ndarray]]:
    results = {}
    for config, emulator in emulators.items():
        seconds = time_rose_predictions(
            emulator,
            testing_rows,
            timing_repeats,
            timing_inner_loops,
            emulated_smatrix_all_channels,
            cross_section_from_smatrix,
            angles_rad,
        )
        train_xs = np.asarray(
            [
                cross_section_from_smatrix(
                    emulator,
                    row,
                    *emulated_smatrix_all_channels(emulator, row),
                    angles_rad,
                )
                for row in training_rows
            ]
        )
        test_xs = np.asarray(
            [
                cross_section_from_smatrix(
                    emulator,
                    row,
                    *emulated_smatrix_all_channels(emulator, row),
                    angles_rad,
                )
                for row in testing_rows
            ]
        )
        results[config] = comparison_result(
            train_xs,
            test_xs,
            fom_train_xs,
            fom_test_xs,
            denominator_floor,
            seconds,
        )
    return results


In [ ]:
rho_mesh = emulator.samples.mesh.rho
rose_emulators = build_rose_emulators(
    BASIS_SIZES,
    ROSE_EIM_SIZES,
    L_MAX,
    len(parameter_names),
    emulator,
    train_rows,
    rho_mesh,
    ANGLES_RAD,
    6 * np.pi,
    bench_full_ws,
    bench_full_ws_so,
)
reference_sae = rose_emulators[(BASIS_SIZES[0], ROSE_EIM_SIZES[0])]
fom_train_xs, fom_test_xs = evaluate_fom_cross_sections(
    reference_sae,
    train_rows,
    test_rows,
    ANGLES_RAD,
)
train_cases = parameter_dicts(train_rows, parameter_names)
test_cases = parameter_dicts(test_rows, parameter_names)
default_key = (DEFAULT_BASIS_SIZE, DEFAULT_COMPRESSION_SIZE)
old_v2_results = evaluate_old_lrom(
    emulator,
    DEFAULT_BASIS_SIZE,
    DEFAULT_COMPRESSION_SIZE,
    ANGLES_DEG,
    test_cases,
    fom_test_xs,
    ERROR_DENOMINATOR_FLOOR,
    TIMING_REPEATS,
    TIMING_INNER_LOOPS,
)
lrom_results, ls_results, default_predictor = evaluate_lrom_grid(
    emulator,
    BASIS_SIZES,
    LROM_PREDICTOR_COUNTS,
    default_key,
    ANGLES_DEG,
    train_rows,
    test_rows,
    train_cases,
    test_cases,
    fom_train_xs,
    fom_test_xs,
    ERROR_DENOMINATOR_FLOOR,
    TIMING_REPEATS,
    TIMING_INNER_LOOPS,
)
archive_lrom_results = lrom_results
rose_results = evaluate_rose_grid(
    rose_emulators,
    train_rows,
    test_rows,
    ANGLES_RAD,
    fom_train_xs,
    fom_test_xs,
    ERROR_DENOMINATOR_FLOOR,
    TIMING_REPEATS,
    TIMING_INNER_LOOPS,
)

assert len(reference_sae.rbes) == L_MAX + 1
assert fom_train_xs.shape == (N_TRAIN, ANGLES_DEG.size)
assert fom_test_xs.shape == (N_TEST, ANGLES_DEG.size)
assert np.all(np.isfinite(fom_train_xs)) and np.all(fom_train_xs >= -1e-12)
assert np.all(np.isfinite(fom_test_xs)) and np.all(fom_test_xs >= -1e-12)
assert default_predictor is not None
for collection in (lrom_results, rose_results, ls_results):
    for result in collection.values():
        for values in result.values():
            assert np.all(np.isfinite(values))
assert set(lrom_results) == {
    (n_phi, predictor_count)
    for n_phi in BASIS_SIZES
    for predictor_count in LROM_PREDICTOR_COUNTS
}
assert set(rose_results) == {
    (n_phi, n_u)
    for n_phi in BASIS_SIZES
    for n_u in ROSE_EIM_SIZES
}
assert set(default_predictor) == set(section3_predictor)
for channel in default_predictor:
    assert np.array_equal(
        default_predictor[channel].selected_indices,
        section3_predictor[channel].selected_indices,
    )
assert np.median(
    lrom_results[default_key]["test_median_over_angle_error"]
) < np.median(
    old_v2_results[default_key]["test_median_over_angle_error"]
)


## 5. Representative Cross-Section Predictions

In [ ]:
def select_alpha_cases(
    lrom_error: np.ndarray,
    rose_error: np.ndarray,
    count: int,
) -> list[int]:
    lrom_rank = np.argsort(np.argsort(lrom_error))
    rose_rank = np.argsort(np.argsort(rose_error))
    ordered = np.argsort(0.5 * (lrom_rank + rose_rank))
    positions = np.linspace(
        0,
        len(ordered) - 1,
        count + 2,
        dtype=int,
    )[1:-1]
    return [int(ordered[position]) for position in positions]


def alpha_selection_table(
    labels: tuple[str, ...],
    indices: list[int],
    test_ids: np.ndarray,
    testing_rows: np.ndarray,
    parameter_names: tuple[str, ...],
) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "selection": label,
                "case_id": test_ids[index],
                **{
                    name: float(testing_rows[index, column])
                    for column, name in enumerate(parameter_names)
                },
            }
            for label, index in zip(labels, indices)
        ]
    )


def representative_cross_section_figure(
    angles_deg: np.ndarray,
    selected_indices: list[int],
    selected_labels: tuple[str, ...],
    test_ids: np.ndarray,
    fom_xs: np.ndarray,
    ls_xs: np.ndarray,
    rose_xs: np.ndarray,
    lrom_xs: np.ndarray,
    ls_label: str,
) -> Figure:
    fig, axes = plt.subplots(
        1,
        len(selected_indices),
        figsize=(15.0, 4.2),
        sharey=True,
    )
    for ax, index, label in zip(
        axes,
        selected_indices,
        selected_labels,
    ):
        ax.semilogy(angles_deg, fom_xs[index], color="black", label="FOM")
        ax.semilogy(
            angles_deg,
            ls_xs[index],
            color="tab:blue",
            label=ls_label,
        )
        ax.semilogy(
            angles_deg,
            rose_xs[index],
            "--",
            color="tab:red",
            label="ROSE",
        )
        ax.semilogy(
            angles_deg,
            lrom_xs[index],
            ":",
            color="#E6AB02",
            lw=2.2,
            label="LROM",
        )
        ax.set_title(f"{label}: {test_ids[index]}")
        ax.set_xlabel("angle [deg]")
    axes[0].set_ylabel(r"d$\sigma$/d$\Omega$ [mb/sr]")
    axes[0].legend()
    fig.tight_layout()
    return fig


def cross_section_error_figure(
    angles_deg: np.ndarray,
    selected_indices: list[int],
    selected_labels: tuple[str, ...],
    test_ids: np.ndarray,
    fom_xs: np.ndarray,
    ls_xs: np.ndarray,
    rose_xs: np.ndarray,
    lrom_xs: np.ndarray,
    ls_label: str,
    denominator_floor: float,
    plotting_floor: float,
) -> Figure:
    fig, axes = plt.subplots(
        1,
        len(selected_indices),
        figsize=(15.0, 4.0),
        sharey=True,
    )
    methods = (
        (ls_xs, ls_label, "tab:blue", "-"),
        (rose_xs, "ROSE", "tab:red", "--"),
        (lrom_xs, "LROM", "#E6AB02", ":"),
    )
    for ax, index, label in zip(
        axes,
        selected_indices,
        selected_labels,
    ):
        for values, method, color, style in methods:
            errors = pointwise_relative_error(
                values[index],
                fom_xs[index],
                denominator_floor,
            )
            ax.semilogy(
                angles_deg,
                np.maximum(errors, plotting_floor),
                style,
                color=color,
                label=method,
            )
        ax.set_title(f"{label}: {test_ids[index]}")
        ax.set_xlabel("angle [deg]")
    axes[0].set_ylim(bottom=plotting_floor)
    axes[0].set_ylabel("relative cross-section error")
    axes[0].legend()
    fig.tight_layout()
    return fig


default_key = (DEFAULT_BASIS_SIZE, DEFAULT_COMPRESSION_SIZE)
selected_indices = select_alpha_cases(
    lrom_results[default_key]["test_median_over_angle_error"],
    rose_results[default_key]["test_median_over_angle_error"],
    3,
)
selected_labels = (
    "alpha selection A",
    "alpha selection B",
    "alpha selection C",
)
alpha_cases = alpha_selection_table(
    selected_labels,
    selected_indices,
    test_ids,
    test_rows,
    parameter_names,
)
display(alpha_cases)

# FIGURE: representative-cross-sections
figure = representative_cross_section_figure(
    ANGLES_DEG,
    selected_indices,
    selected_labels,
    test_ids,
    fom_test_xs,
    ls_results[DEFAULT_BASIS_SIZE]["test_xs"],
    rose_results[default_key]["test_xs"],
    lrom_results[default_key]["test_xs"],
    LS_LABEL,
)
plt.show()

# FIGURE: cross-section-errors
figure = cross_section_error_figure(
    ANGLES_DEG,
    selected_indices,
    selected_labels,
    test_ids,
    fom_test_xs,
    ls_results[DEFAULT_BASIS_SIZE]["test_xs"],
    rose_results[default_key]["test_xs"],
    lrom_results[default_key]["test_xs"],
    LS_LABEL,
    ERROR_DENOMINATOR_FLOOR,
    PLOTTING_FLOOR,
)
plt.show()


## 6. Cross-Section Error Distributions

In [ ]:
def error_distribution_figure(
    ls_result: dict[str, np.ndarray],
    rose_result: dict[str, np.ndarray],
    lrom_result: dict[str, np.ndarray],
    plotting_floor: float,
) -> Figure:
    categories = (
        ("LS-projected", ls_result),
        ("ROSE", rose_result),
        ("LROM", lrom_result),
    )
    positions = np.arange(1, len(categories) + 1)
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    for center_pos, (_, result) in zip(positions, categories):
        for side, key, color, alpha in (
            (-1, "train_median_over_angle_error", "tab:blue", 0.75),
            (+1, "test_median_over_angle_error", "tab:orange", 0.55),
        ):
            values = np.log10(
                np.clip(result[key], plotting_floor, None)
            )
            parts = ax.violinplot(
                [values],
                positions=[center_pos],
                widths=0.85,
                showextrema=False,
            )
            body = parts["bodies"][0]
            vertices = body.get_paths()[0].vertices
            vertices[:, 0] = (
                np.minimum(vertices[:, 0], center_pos)
                if side < 0
                else np.maximum(vertices[:, 0], center_pos)
            )
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_alpha(alpha)
            ax.scatter(
                center_pos + 0.07 * side,
                np.median(values),
                marker="D",
                color=color,
                edgecolor="black",
                zorder=6,
            )
    ax.set_xticks(positions, [label for label, _ in categories])
    ax.set_ylabel("log10 median pointwise relative error")
    ax.legend(
        handles=[
            Patch(
                facecolor="tab:blue",
                alpha=0.75,
                edgecolor="black",
                label="train",
            ),
            Patch(
                facecolor="tab:orange",
                alpha=0.55,
                edgecolor="black",
                label="test",
            ),
            Line2D(
                [],
                [],
                marker="D",
                linestyle="None",
                color="0.3",
                label="median",
            ),
        ]
    )
    fig.tight_layout()
    return fig


# FIGURE: error-violins
figure = error_distribution_figure(
    ls_results[DEFAULT_BASIS_SIZE],
    rose_results[default_key],
    lrom_results[default_key],
    PLOTTING_FLOOR,
)
plt.show()


## 7. Basis and Operator-Size Comparison

In [ ]:
def summary_results_table(
    rose_results: dict[tuple[int, int], dict[str, np.ndarray]],
    lrom_results: dict[tuple[int, int], dict[str, np.ndarray]],
) -> pd.DataFrame:
    rows = []
    for method, symbol, results in (
        ("ROSE", "n_U", rose_results),
        ("LROM", "K", lrom_results),
    ):
        for (n_phi, compression), result in results.items():
            rows.append(
                {
                    "method": method,
                    "n_phi": n_phi,
                    "compression symbol": symbol,
                    "compression": compression,
                    "median pointwise test error": float(
                        np.median(
                            result["test_median_over_angle_error"]
                        )
                    ),
                    "median maximum-over-angle error": float(
                        np.median(
                            result["test_maximum_over_angle_error"]
                        )
                    ),
                    "median online time [s]": float(
                        np.median(result["test_seconds"])
                    ),
                }
            )
    return pd.DataFrame(rows).sort_values(
        ["method", "n_phi", "compression"]
    )


summary_table = summary_results_table(rose_results, lrom_results)
summary_table


## 8. Accuracy Versus Online Time

In [ ]:
def accuracy_time_figure(
    rose_results: dict[tuple[int, int], dict[str, np.ndarray]],
    lrom_results: dict[tuple[int, int], dict[str, np.ndarray]],
    basis_sizes: tuple[int, ...],
    compression_values: tuple[int, ...],
    plotting_floor: float,
    error_reference: float,
    hourly_throughput: int,
) -> Figure:
    fig, ax = plt.subplots(figsize=(12.5, 5.4))
    palette = ("tab:blue", "tab:green", "tab:red")
    colors = dict(zip(basis_sizes, palette))
    marker_sizes = dict(zip(compression_values, (16, 28, 44)))
    for (n_phi, n_u), result in rose_results.items():
        ax.scatter(
            float(np.median(result["test_seconds"])),
            max(
                float(
                    np.median(
                        result["test_median_over_angle_error"]
                    )
                ),
                plotting_floor,
            ),
            marker="s",
            s=marker_sizes[n_u],
            alpha=0.35,
            color=colors[n_phi],
        )
    for (n_phi, predictor_count), result in lrom_results.items():
        ax.scatter(
            float(np.median(result["test_seconds"])),
            max(
                float(
                    np.median(
                        result["test_median_over_angle_error"]
                    )
                ),
                plotting_floor,
            ),
            marker="o",
            s=marker_sizes[predictor_count],
            alpha=0.35,
            facecolors="none",
            edgecolors=colors[n_phi],
        )
    ax.axhline(error_reference, color="0.25", linestyle="--")
    ax.axvline(3600 / hourly_throughput, color="0.45", linestyle=":")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("online time per sample [s]")
    ax.set_ylabel("median pointwise relative error")
    ax.set_title("Computational Accuracy versus Time")

    handles = [
        Line2D(
            [],
            [],
            marker="s",
            linestyle="None",
            color="black",
            markerfacecolor="black",
            label="ROSE",
        ),
        Line2D(
            [],
            [],
            marker="o",
            linestyle="None",
            color="black",
            markerfacecolor="none",
            label="LROM",
        ),
    ]
    handles.extend(
        Line2D(
            [],
            [],
            marker="o",
            linestyle="None",
            color=color,
            markerfacecolor=color,
            label=f"n_phi={n_phi}",
        )
        for n_phi, color in colors.items()
    )
    handles.extend(
        Line2D(
            [],
            [],
            marker="o",
            markersize=np.sqrt(size),
            linestyle="None",
            color="0.4",
            markerfacecolor="0.7",
            label=f"compression control={compression}",
        )
        for compression, size in marker_sizes.items()
    )
    handles.extend(
        [
            Line2D(
                [],
                [],
                color="0.25",
                linestyle="--",
                label=f"{error_reference:.2f} median pointwise error",
            ),
            Line2D(
                [],
                [],
                color="0.45",
                linestyle=":",
                label="one million evaluations/hour",
            ),
        ]
    )
    ax.legend(
        handles=handles,
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        fontsize=8,
    )
    fig.subplots_adjust(right=0.72)
    return fig


# FIGURE: cat-plot
figure = accuracy_time_figure(
    rose_results,
    lrom_results,
    BASIS_SIZES,
    ROSE_EIM_SIZES,
    PLOTTING_FLOOR,
    0.10,
    1_000_000,
)
plt.show()


## 9. Validation Summary

In [ ]:
def validation_results_table(
    training_shape: tuple[int, ...],
    testing_shape: tuple[int, ...],
    l_max: int,
    half_width: float,
    basis_sizes: tuple[int, ...],
    rose_eim_sizes: tuple[int, ...],
    lrom_predictor_counts: tuple[int, ...],
    old_error: np.ndarray,
    archive_error: np.ndarray,
    python_version: str,
    platform_name: str,
    lrom_version: str,
) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"check": "training rows", "value": training_shape},
            {"check": "testing rows", "value": testing_shape},
            {"check": "ROSE/LROM shared rows", "value": True},
            {
                "check": "partial waves",
                "value": tuple(range(l_max + 1)),
            },
            {"check": "parameter half-width", "value": half_width},
            {"check": "basis sizes", "value": basis_sizes},
            {"check": "ROSE EIM sizes", "value": rose_eim_sizes},
            {
                "check": "LROM predictor counts",
                "value": lrom_predictor_counts,
            },
            {
                "check": "LROM predictor method",
                "value": "effective-interaction",
            },
            {
                "check": "old v2 default median pointwise error",
                "value": float(np.median(old_error)),
            },
            {
                "check": "archive LROM default median pointwise error",
                "value": float(np.median(archive_error)),
            },
            {"check": "LROM version", "value": lrom_version},
            {"check": "Python", "value": python_version},
            {"check": "platform", "value": platform_name},
        ]
    )


# TABLE: validation-summary
validation_summary = validation_results_table(
    train_rows.shape,
    test_rows.shape,
    L_MAX,
    HALF_WIDTH,
    BASIS_SIZES,
    ROSE_EIM_SIZES,
    LROM_PREDICTOR_COUNTS,
    old_v2_results[default_key]["test_median_over_angle_error"],
    archive_lrom_results[default_key]["test_median_over_angle_error"],
    sys.version.split()[0],
    platform.platform(),
    lrom.__version__,
)
validation_summary
